<a href="https://colab.research.google.com/github/firdoushkhilji/firdoushkhilji-7006SCN_FK_16943920/blob/main/Notebooks/Task1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [39]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [40]:
import os
os.listdir('/content/drive/MyDrive/')

['TASK_DATASET', 'Colab Notebooks']

In [41]:
os.listdir('/content/drive/MyDrive/TASK_DATASET/')

['epd_snomed_202603.csv']

In [42]:
#import and start pyspark
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("NHS_Prescription_Cost_Prediction") \
    .getOrCreate()

print('Spark Successfully Started!')

Spark Successfully Started!


In [49]:
#READ CSV FILE USING SPARK
df = spark.read.csv('/content/drive/MyDrive/TASK_DATASET/epd_snomed_202603.csv', header=True, inferSchema=True)

In [50]:
print('Top 20 Records - \n')
df.show(20)

Top 20 Records - 

+-------------------+--------------------+--------------------+--------------------+--------+--------------------+--------+-------------------+-------------+-------------------+---------------+-------------------+---------+--------+---------------------------+----------------------+---------------------+---------------------+---------------------+--------+-----+--------------+---------+-----+-----------+------------+-----------------+
|         YEAR_MONTH|REGIONAL_OFFICE_NAME|REGIONAL_OFFICE_CODE|            ICB_NAME|ICB_CODE|            PCO_NAME|PCO_CODE|      PRACTICE_NAME|PRACTICE_CODE|          ADDRESS_1|      ADDRESS_2|          ADDRESS_3|ADDRESS_4|POSTCODE|BNF_CHEMICAL_SUBSTANCE_CODE|BNF_CHEMICAL_SUBSTANCE|BNF_PRESENTATION_CODE|BNF_PRESENTATION_NAME|BNF_CHAPTER_PLUS_CODE|QUANTITY|ITEMS|TOTAL_QUANTITY|ADQ_USAGE|  NIC|ACTUAL_COST|UNIDENTIFIED|      SNOMED_CODE|
+-------------------+--------------------+--------------------+--------------------+--------+----------

In [51]:
print('Description of Dataset -\n')
df.printSchema()

Description of Dataset -

root
 |-- YEAR_MONTH: timestamp (nullable = true)
 |-- REGIONAL_OFFICE_NAME: string (nullable = true)
 |-- REGIONAL_OFFICE_CODE: string (nullable = true)
 |-- ICB_NAME: string (nullable = true)
 |-- ICB_CODE: string (nullable = true)
 |-- PCO_NAME: string (nullable = true)
 |-- PCO_CODE: string (nullable = true)
 |-- PRACTICE_NAME: string (nullable = true)
 |-- PRACTICE_CODE: string (nullable = true)
 |-- ADDRESS_1: string (nullable = true)
 |-- ADDRESS_2: string (nullable = true)
 |-- ADDRESS_3: string (nullable = true)
 |-- ADDRESS_4: string (nullable = true)
 |-- POSTCODE: string (nullable = true)
 |-- BNF_CHEMICAL_SUBSTANCE_CODE: string (nullable = true)
 |-- BNF_CHEMICAL_SUBSTANCE: string (nullable = true)
 |-- BNF_PRESENTATION_CODE: string (nullable = true)
 |-- BNF_PRESENTATION_NAME: string (nullable = true)
 |-- BNF_CHAPTER_PLUS_CODE: string (nullable = true)
 |-- QUANTITY: double (nullable = true)
 |-- ITEMS: integer (nullable = true)
 |-- TOTAL_QUANT

In [52]:
#Dataset Requirement is more than 10million rows
rows_count = df.count()

print(f"Total Records in Dataset - {rows_count:,}")
print(f"Total Columns/Features - {len(df.columns):,}")

if rows_count >= 10000000:
      print('Dataset valid since rows is greater than/equal to 10 million.')
else:
      print('Dataset not Valid since rows is lesser than 10 million.')


Total Records in Dataset - 18,364,409
Total Columns/Features - 27
Dataset valid since rows is greater than/equal to 10 million.


In [53]:
#size of dataset file
import os
file_path = '/content/drive/MyDrive/TASK_DATASET/epd_snomed_202603.csv'
file_size = os.path.getsize(file_path)
print(f'File Size: {file_size / (1024**3):.2f} GB')

File Size: 7.15 GB


7006SCN - Machine Learning and Big Data - 2526MAYJUL

Task 1 - Problem Statement & Dataset Description

## Problem Statement
Every Year the NHS spends over £9 Billion on Prescriptions alone.
Predicting how much a prescription will cost remains inconsistent over regions, GP Practices and Drug Types.

The same drug can cost dramatically different to the NHS, based on where it has been prescribed in England. These patterns are hidden amongst 18 million records hidden to human eye but detectable by Machine Learning.

This project predicts the Actual Cost(ACTUAL_COST) of NHS prescriptions using a PySpark ML pipeline across 18 million records.

> Can we predict the actual cost of an NHS prescription based on the drug type, GP practice, region and quantity?

###Answering this question could help the NHS:
- Identify GP practices with unusually high prescribing costs
- Plan regional budgets more accurately
- Flag anomalies for investigation before they become expensive
- Save millions annually through smarter resource allocation

##Dataset
To answer this question, we use the NHS English Prescribing Dataset with SNOMED codes — published monthly by the NHS Business Services Authority (NHSBSA).

### Big Data Justification
- 18,364,409 rows (>10 million rows)
- 27 columns (>10 columns)
- ML Type: Regression
- 7.15 GB — too large for Excel or standard Pandas
- Requires distributed computing via Apache PySpark

###Dataset Source
####Source: NHS Business Services Authority (NHSBSA)
####Period: March 2026
####Rows: 18,364,409
####Columns: 27
####Target Variable: ACTUAL_COST (£)
####URL: https://opendata.nhsbsa.net/dataset/english-prescribing-dataset-epd-with-snomed-code/resource/6b52baa3-c550-4b1d-8cd2-46d40d840353
####File Size: 7.15 GB

This dataset cannot be opened in Excel, processed in standard Pandas, or analysed on a single CPU efficiently. This is precisely why Apache PySpark is used.

## Ethical & Licensing Note
####Licence:	Open Government Licence 3.0 (United Kingdom)

This dataset is published under the NHSBSA Open Data Licence
(OGL v3.0) and is free to use, share and adapt with credit.

Crucially, the dataset contains NO patient-level identifiers.
There are no names, NHS numbers, dates of birth or any
personally identifiable information. Every row represents
an aggregated prescription count at GP practice level —
not an individual patient record.

This means no ethics approval is required to use this data
for research or educational purposes.

Bias consideration: Rural GP practices may show different
cost patterns to urban ones — not due to poor prescribing,
but due to population demographics and drug availability.
This will be examined during model evaluation in Task 5.

Credit: "NHSBSA Copyright 2025"
